# NetraEdge - Liveness Detection Training
## LivenessCNN: Real vs Print vs Screen | GPU: T4 | ~1 hour

In [ ]:
#@title Step 1: Install + GPU Check
!pip install -q onnx tqdm scikit-learn onnxscript
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())
else:
    print('WARNING: No GPU! Runtime > Change runtime type > GPU')

In [ ]:
#@title Step 2: Load LFW + Generate Spoof Data
from sklearn.datasets import fetch_lfw_people
from PIL import Image
import numpy as np
import os, random

print('Loading LFW via sklearn...')
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
print('Loaded:', lfw.images.shape[0], 'images,', len(lfw.target_names), 'identities')

output_dir = 'lfw_persons'
os.makedirs(output_dir, exist_ok=True)
for i, name in enumerate(lfw.target_names):
    mask = lfw.target == i
    if mask.sum() < 10:
        continue
    person_dir = os.path.join(output_dir, name.replace(' ', '_'))
    os.makedirs(person_dir, exist_ok=True)
    for j, idx in enumerate(np.where(mask)[0]):
        img = Image.fromarray(lfw.images[idx].astype(np.uint8))
        img = img.resize((112, 112), Image.BILINEAR)
        img.save(os.path.join(person_dir, str(j).zfill(4) + '.jpg'))

persons = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d))]
print('Saved', len(persons), 'persons to', output_dir)

In [ ]:
#@title Step 3: LivenessCNN Architecture
import torch.nn as nn
import torch.nn.functional as F

class LBlock(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class LivenessCNN(nn.Module):
    def __init__(self, nc=3, dp=0.3):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            LBlock(32, 64), LBlock(64, 128, 2),
            LBlock(128, 256, 2), LBlock(256, 256, 2))
        self.cls = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dp), nn.Linear(256, 64), nn.ReLU(True),
            nn.Dropout(dp * 0.5), nn.Linear(64, nc))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        return self.cls(self.feat(x))

livmodel = LivenessCNN(3)
out = livmodel(torch.randn(2, 3, 112, 112))
nparams = sum(p.numel() for p in livmodel.parameters())
print('LivenessCNN:', nparams, 'params, output:', out.shape)

In [ ]:
#@title Step 4: Synthetic Liveness Dataset
from torch.utils.data import Dataset, DataLoader

class SynLiveness(Dataset):
    def __init__(self, root='lfw_persons', ns=3000):
        self.imgs = []
        for r, ds, fs in os.walk(root):
            for f in fs:
                if f.endswith('.jpg'):
                    self.imgs.append(os.path.join(r, f))
        self.ns = ns
        print('Found', len(self.imgs), 'source images')
    def __len__(self):
        return self.ns
    def __getitem__(self, idx):
        path = random.choice(self.imgs)
        label = idx % 3
        img = Image.open(path).convert('RGB').resize((112, 112))
        arr = np.array(img).astype(np.float32) / 255.0
        if label == 1:
            arr = np.clip(arr * 1.1 + 0.04, 0, 1)
            k = np.ones(9) / 9
            arr = np.stack([np.convolve(arr[:,:,c].flatten(), k, mode='same').reshape(112,112) for c in range(3)], -1)
        elif label == 2:
            for y in range(0, 112, 2):
                arr[y,:,:] *= 0.85
            arr[:,:,2] *= 1.1
            arr = arr * 0.8 + 0.12
        arr = np.clip(arr + np.random.normal(0, 0.02, arr.shape), 0, 1)
        if random.random() > 0.5:
            arr = np.flip(arr, axis=1).copy()
        tensor = torch.from_numpy(arr).permute(2, 0, 1).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        return (tensor - mean) / std, label

train_ds = SynLiveness('lfw_persons', 3000)
val_ds = SynLiveness('lfw_persons', 600)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print('Train:', len(train_ds), 'Val:', len(val_ds))

In [ ]:
#@title Step 5: Train 15 Epochs
import time
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
livmodel = LivenessCNN(3).to(DEVICE)
weights = torch.tensor([1.0, 1.5, 1.5]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(livmodel.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
best_acc = 0

for epoch in range(15):
    t0 = time.time()
    livmodel.train()
    train_correct = 0
    train_total = 0
    for images, labels in tqdm(train_loader, desc='Epoch ' + str(epoch + 1) + '/15'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(livmodel(images), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(livmodel.parameters(), 5.0)
        optimizer.step()
        train_correct += (livmodel(images).argmax(1) == labels).sum().item()
        train_total += labels.size(0)
    livmodel.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            val_correct += (livmodel(images).argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    train_acc = 100.0 * train_correct / train_total
    val_acc = 100.0 * val_correct / val_total
    scheduler.step()
    elapsed = int(time.time() - t0)
    print('Epoch ' + str(epoch + 1) + ': Train ' + str(round(train_acc, 1)) + '% Val ' + str(round(val_acc, 1)) + '% ' + str(elapsed) + 's')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(livmodel.state_dict(), 'liv_best.pt')
        print('  Saved best model (' + str(round(val_acc, 1)) + '%)')
print('Training complete. Best: ' + str(round(best_acc, 1)) + '%')

In [ ]:
#@title Step 6: Export ONNX
livmodel.cpu().eval()
torch.onnx.export(
    livmodel, torch.randn(1, 3, 112, 112),
    'liveness_detector.onnx',
    input_names=['input'],
    output_names=['output'],
    opset_version=13,
    dynamo=False)
sz = os.path.getsize('liveness_detector.onnx') / 1e6
print('Exported: liveness_detector.onnx (' + str(round(sz, 1)) + 'MB)')
print('Download and place in F:/PROJECTS/NetraEdge/models/')